# 04. XGBoost Training

Training XGBoost Regressor using all engineered features.

In [ ]:
import pandas as pd
import xgboost as xgb
import joblib
import os
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

df = pd.read_csv("../backend/data/processed/engineered_features.csv")
df["date"] = pd.to_datetime(df["date"])

features = ["price_lag_7", "price_lag_14", "price_lag_30", "rolling_mean_7", "rolling_mean_30", "rolling_std_7", "day_of_week", "month", "temperature", "rainfall", "ndvi_proxy"]

def train_xgboost(commodity):
    data = df[df["commodity"] == commodity].sort_values("date")
    X = data[features]
    y = data["price"]
    
    split = int(len(data) * 0.8)
    X_train, X_test = X.iloc[:split], X.iloc[split:]
    y_train, y_test = y.iloc[:split], y.iloc[split:]
    
    model = xgb.XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8)
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], early_stopping_rounds=50, verbose=False)
    
    pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, pred)
    mape = mean_absolute_percentage_error(y_test, pred)
    print(f"{commodity} XGBoost - MAE: {mae:.2f}, MAPE: {mape:.4f}")
    
    joblib.dump(model, f"../backend/models/saved/xgboost_{commodity}.pkl")
    
    # Feature Importance
    plt.figure(figsize=(10, 5))
    importances = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
    importances.plot(kind='bar')
    plt.title(f"XGBoost Feature Importance - {commodity.capitalize()}")
    plt.tight_layout()
    plt.savefig(f"04_xgboost_{commodity}_feat_imp.png")
    plt.close()

for c in ["onion", "potato", "tomato"]:
    train_xgboost(c)